# Climate Sentiment

# Setup

In [3]:
import torch
import random
import numpy as np

def set_seed(seed: int = 42):
    random.seed(seed)  # Python's built-in random module
    np.random.seed(seed)  # NumPy
    torch.manual_seed(seed)  # PyTorch CPU
    torch.cuda.manual_seed(seed)  # PyTorch GPU (if available)
    torch.cuda.manual_seed_all(seed)  # Multi-GPU
    torch.backends.cudnn.deterministic = True  # Ensure deterministic behavior
    torch.backends.cudnn.benchmark = False  # Disable benchmark mode for reproducibility

### SET SEED TO RANDOM NUMBER
seed_number = 2546204
set_seed(seed_number)

# Setup: loading the data

In [5]:
import os
import pandas as pd

os.mkdir('./outputs')

In [6]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
from sklearn.model_selection import train_test_split 

The dataset classifies paragraphs taken from corporate disclosures that discuss climate-related issues. It classifiers them into "risk" (0), "neutral" (1) or "opportunity" (2) representing the sentiment of the paragraph.

In [8]:
cache_dir = "./data_cache"

# load the original training set from HuggingFace
train_dataset = load_dataset(
    "climatebert/climate_sentiment",
    split="train",
    cache_dir=cache_dir,
)

# we're going to create a new validation set by splitting the data
dataset_splits = train_dataset.train_test_split(test_size=0.2)
train_dataset = dataset_splits["train"]
val_dataset = dataset_splits["test"]

train_texts = np.array(train_dataset["text"])
val_texts = np.array(val_dataset["text"])

train_labels = np.array(train_dataset["label"])
val_labels = np.array(val_dataset["label"])

print(f"Training dataset with {len(train_texts)} instances loaded")
print(f"Development/validation dataset with {len(val_texts)} instances loaded")

# save gold labels to file
pd.DataFrame(val_labels).to_csv('./outputs/val_labels.csv')

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/320 [00:00<?, ? examples/s]

Training dataset with 800 instances loaded
Development/validation dataset with 200 instances loaded


In this notebook, we're going to build three different classifiers for this dataset, then compare how they work, and analyse the results. We are going to start by implementing a naïve Bayes classifier from scratch. 

We are going to begin by initialising some useful variables and doing some very simple pre-processing using CountVectorizer.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk import word_tokenize

num_classes = 3

def preprocess(train_texts):
    vectorizer = CountVectorizer(ngram_range=(2,2), tokenizer=word_tokenize)
    X = vectorizer.fit_transform(train_texts).toarray()
    num_features = X.shape[1]

    X_val = vectorizer.transform(val_texts).toarray()

    return X, X_val, vectorizer, num_features

X, X_val, vectorizer, num_features = preprocess(train_texts)

C:\Users\snand\AppData\Local\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


## A - Understanding Class Distribution

Before training classifiers, it’s important to understand the dataset’s label distribution. We compute *class priors* — the probability of each sentiment class occurring in the training set:
$p(y_n = c)$ for each class label $c$, where $y_n$ is the class label of document $n$ 

We have not used the Sklearn implementation to do this, but implemented it ourself, e.g., using Numpy functions. The function outputs the class priors as a list or Numpy array containing the probabilities.

In [12]:
def compute_class_priors(texts, labels):
    priors = np.zeros(num_classes)
    
    total = len(train_labels)
    for c in range(num_classes):
        priors[c] = np.sum(train_labels == c)/total
    
    return priors


class_priors = compute_class_priors(train_texts, train_labels)
print(class_priors)

pd.DataFrame(class_priors).to_csv('./outputs/a_class_priors.csv')

[0.34375 0.40625 0.25   ]


## B - Feature Likelihoods with Naive Bayes

After computing class priors, the next step is to estimate how strongly each **n‑gram feature** is associated with each sentiment class. This is the foundation of the **Naive Bayes classifier**.

### Why Feature Likelihoods Matter
For each class $c$, we want to know the probability that a given n‑gram $w$ appears in documents of that class:
$p(x_{ni} = w | y_n = c)$ 

This tells us how characteristic a word or phrase is of a particular sentiment (risk, opportunity, or neutral).

### Implementation Details
We compute likelihoods manually using NumPy, applying Laplace smoothing (+1 to every count) to avoid zero probabilities:

In [14]:
def compute_feature_likelihoods(X, labels):

    num_classes = len(np.unique(labels))
    num_features = X.shape[1]
    
    likelihoods = np.ones((num_features, num_classes))  # a 2D numpy array where you can store the likelihoods. Note that all values are initialised to one.

    alpha = 1.0 # for laplace smoothing
    
    for c in range(num_classes):
        X_c = X[labels == c] # select rows in X for class c
        class_counts = X_c.sum(axis = 0) # total count of each feature (column) for class c
        smoothing = class_counts + alpha # apply Laplace smoothing : add alpha
        likelihoods[:, c] = smoothing / smoothing.sum()

    return likelihoods

likelihoods = compute_feature_likelihoods(X, train_labels)

pd.DataFrame(likelihoods).to_csv('./outputs/b_likelihoods.csv')

## Naive Bayes Classification

With priors and likelihoods in place, we can classify new documents. The Naive Bayes classifier computes the log joint probability of each class given the features:
$$
\log p(y \mid x) \propto \log p(y) + \sum_i x_i \log p(x_i \mid y
$
We normalize using `logsumexp` to avoid numerical instability.$


In [16]:
from scipy.special import logsumexp


def NB_classify(class_priors, likelihoods, X_val):

    predictions = np.zeros((X_val.shape[0], num_classes))  # an empty numpy array to store the predictions in

    sum_of_log_likelihoods = X_val.dot(np.log(likelihoods))
    log_joint_prob = sum_of_log_likelihoods + np.log(class_priors)[None, :]
    for n, doc in enumerate(X_val):
        predictions[n, :] = log_joint_prob[n]
        predictions[n, :] -= logsumexp(predictions[n, :])
    return predictions

predictions = NB_classify(class_priors, likelihoods, X_val)
print(predictions)

[[-1.08203883e+01 -6.77715997e+00 -1.16016707e-03]
 [-1.12660519e+01 -1.12606419e+01 -2.56701118e-05]
 [-3.01652477e+00 -5.49983858e-02 -5.39434037e+00]
 [-9.93086965e+00 -4.98610083e-05 -1.36246461e+01]
 [-3.35418860e-01 -2.17880144e+00 -1.76151520e+00]
 [-3.81419341e-10 -2.17443647e+01 -2.45770359e+01]
 [-2.73214733e+01 -1.36424205e-12 -3.50390524e+01]
 [-4.29806388e+01 -4.01870313e+01  0.00000000e+00]
 [-1.12194937e+01 -2.33845958e-03 -6.06518976e+00]
 [-1.15942143e+00 -4.06856373e-01 -3.88282395e+00]
 [-5.68434189e-14 -3.07578531e+01 -3.53652063e+01]
 [-1.74586343e+01 -2.62557478e-08 -2.31855951e+01]
 [-2.68975942e-01 -1.44522355e+00 -8.83891377e+00]
 [-6.76700255e+00 -1.31553257e+00 -3.14002119e-01]
 [-5.57291742e-08 -1.67027622e+01 -3.37233922e+01]
 [-2.11562006e+01 -1.17378349e+01 -7.98656572e-06]
 [-6.03213114e+01 -3.90409140e+01  0.00000000e+00]
 [ 0.00000000e+00 -5.20489905e+01 -6.31381946e+01]
 [ 0.00000000e+00 -3.93572480e+01 -5.82650870e+01]
 [-3.40407210e+01 -5.68434189e-

The output is a probability distribution over classes for each validation document.

### Model Evaluation
Finally, we evaluate performance using accuracy:

In [18]:
from sklearn.metrics import accuracy_score

accuracy_score(val_labels, np.argmax(predictions, axis=1))

0.755

## C - Interpreting Naive Bayes — Top Features
One of the strengths of the **Naive Bayes classifier** is its interpretability. Since the model is based on feature likelihoods, we can directly inspect which n‑grams are most indicative of a particular class.

For example, for class 0 (which represents climate risk in this dataset), we extract the five n‑grams with the highest likelihood values:

In [20]:
top_features = []

class0_likelihoods = likelihoods[:, 0] # getting likelihoods for class 0
top5_indices = np.argsort(class0_likelihoods)[-5:][::-1] # getting indices of the top 5 highest likelihoods for class 0

feature_names = vectorizer.get_feature_names_out() # use vectorizer to get feature names
top_features = feature_names[top5_indices].tolist() # map indices to actual feature strings

print(top_features)
pd.DataFrame(top_features).to_csv('./outputs/c_top_feats.csv')


['climate change', 'of the', ', and', 'in the', 'to the']


This gives us the top five risk‑related n‑grams, providing insight into the language patterns companies use when framing climate change as a risk. These features are saved for reproducibility. 

## D - Improving the Naive Bayes Classifier

While the baseline Naive Bayes model provides a good starting point, performance can often be improved by refining the **text preprocessing pipeline**. Here, we introduce several enhancements:

* **Unigrams + Bigrams:** Capture both single words and short phrases.
* **Stopword Removal:** Eliminate common words (e.g., “the”, “and”) that add little meaning.
* **Vocabulary Limiting:** Restrict to the top 5,000 features to reduce noise and improve generalization.
* **Binary Features:** Record presence/absence of n‑grams rather than raw counts, which often works better for sentiment tasks.

We then retrain the Naive Bayes classifier with these improved features and evaluate accuracy

In [22]:
from nltk import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

In [23]:
def preprocess_improved(train_texts):
    
    vectorizer = CountVectorizer(
        ngram_range = (1, 2), # updated from (2, 2) - > (1, 2) unigrams + bigrams
        tokenizer = word_tokenize,
        token_pattern = None,
        stop_words = 'english', # removing common stop words
        max_features = 5000, # limiting vocabulary size ## 2000 -> 0.79; 3000 -> 0.8; 5000 -> 0.81
        binary = True
    )
    X = vectorizer.fit_transform(train_texts).toarray()
    num_features = X.shape[1]

    X_val = vectorizer.transform(val_texts).toarray()

    return X, X_val, vectorizer, num_features

X, X_val, vectorizer, num_features = preprocess_improved(train_texts)
class_priors = compute_class_priors(train_texts, train_labels)
likelihoods = compute_feature_likelihoods(X, train_labels)
predictions = NB_classify(class_priors, likelihoods, X_val)
predictions_nb = np.argmax(predictions, axis=1)


pd.DataFrame(predictions_nb).to_csv('./outputs/d_improved_preds.csv')
accuracy_improved = accuracy_score(val_labels, predictions_nb)
print(accuracy_improved)

0.78


## Neural Network Classifier

While Naive Bayes provides a strong baseline, neural networks allow us to capture more complex patterns in text. Here, we implement a **feed‑forward neural network** for climate sentiment classification.

### Tokenization and Data Preparation
We use the **ClimateBERT** tokenizer to convert text into numerical sequences. Each document is padded or truncated to a fixed length (`sequence_length = 64`) for consistency. Data is then batched for training and validation:

### Model Architecture
We design a **feed‑forward text classifier** with the following components:
* **Embedding Layer:** Converts token IDs into dense vectors.
* **Dropout:** Prevents overfitting by randomly dropping units.
* **Two Hidden Layers:** Capture non‑linear relationships.
* **ReLU Activation:** Introduces non‑linearity.
* **Output Layer:** Maps hidden features to sentiment classes.

### Training Strategy
We train the model using **Adam optimizer** with L2 regularization (`weight_decay`) and a sensible learning rate (`1e-3`). Training runs for **10 epochs** to allow convergence.

### Prediction
After training, predictions are made by selecting the class with the highest probability.

In [25]:
set_seed(seed_number)
 
from torch import nn
from torch import optim
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

tokenizer = AutoTokenizer.from_pretrained("climatebert/distilroberta-base-climate-f")  

sequence_length = 64 # updated: giving the model more context  # truncate all docs longer than this. Pad all docs shorter than this.
batch_size = 8 # updated : larger batch size for efficiency

def tokenize_function(examples):
    return tokenizer(
        examples["text"],  # Adjust the key based on your dataset structure
        padding="max_length",  # Ensures equal sequence lengths
        truncation=True,       # Truncates longer sequences
        max_length=sequence_length,        # Adjust as needed
        return_tensors="pt"
    )

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])  # Adjust column names
train_loader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=True)

tokenized_dataset = val_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])  # Adjust column names
val_loader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=False)

class FFTextClassifier(nn.Module):
    
    def __init__(self, vocab_size, sequence_length, num_classes, embedding_size = 128, hidden_size = 128):
        super(FFTextClassifier, self).__init__()

        self.embedding_size = embedding_size
        self.sequence_length = sequence_length

        # Here we just need to construct the components of our network. We don't need to connect them together yet.
        self.embedding_layer = nn.Embedding(vocab_size, embedding_size) # embedding layer
        self.dropout = nn.Dropout(0.3) # added droput

        # Adding 2 hidden layers
        self.hidden_layer1 = nn.Linear(self.embedding_size*sequence_length, hidden_size)  #nn.LSTM(self.embedding_size, hidden_size, bidirectional=True, batch_first=True) # Hidden layer
        self.hidden_layer2 = nn.Linear(hidden_size, hidden_size)
        
        self.activation = nn.ReLU() # Hidden layer
        self.output_layer = nn.Linear(hidden_size, num_classes) # Full connection layer
        
        
    def forward(self, input_words):
        # Input dimensions are:  (batch_size, seq_length)
        embedded_words = self.embedding_layer(input_words)  # (batch_size, seq_length, embedding_size)

        # flatten the sequence of embedding vectors for each document into a single vector.
        #embedded_words = embedded_words.reshape(embedded_words.shape[0], self.sequence_length*self.embedding_size)  #(embedded_words.shape[0], self.sequence_length*self.embedding_size)  # batch_size, seq_length*embedding_size

        flat = embedded_words.view(embedded_words.size(0), -1)
        flat = self.dropout(flat)
        
        hidden1 = self.activation(self.hidden_layer1(flat))
        hidden1 = self.dropout(hidden1)
        hidden2 = self.activation(self.hidden_layer2(hidden1))

        output = self.output_layer(hidden2)

        #z = self.hidden_layer(embedded_words)   # (batch_size, seq_length, hidden_size)
        #h = self.activation(z)                  # (batch_size, seq_length, hidden_size)

        #output = self.output_layer(h)                      # (batch_size, num_classes)

        # Notice we haven't applied a softmax activation to the output layer -- it's not required by Pytorch's loss function.

        return output


    def run_training(self, train_dataloader, dev_dataloader):

        # training hyperparameters
        num_epochs = 10 # updated to 10 epochs instead of 1   
        learning_rate = 1e-3 # updated from 5e5  # learning rate for the gradient descent optimizer, related to the step size
        weight_decay = 1e-5 # added: L2 regularization
        
        loss_fn = nn.CrossEntropyLoss()  # create loss function object
        optimizer = optim.Adam(self.parameters(), lr = learning_rate, weight_decay = weight_decay)  # create the optimizer
        
        dev_losses = []
            
        for e in range(num_epochs):
            # Track performance on the training set as we are learning...
            train_losses = []

            self.train()  # Put the model in training mode.

            for i, batch in enumerate(train_dataloader):
                # Iterate over each batch of data

                optimizer.zero_grad()  # Reset the optimizer

                # Use the model to perform forward inference on the input data.
                # This will run the forward() function.
                output = self(batch['input_ids'])

                # Compute the loss for the current batch of data
                batch_loss = loss_fn(output, batch['label'])

                # Perform back propagation to compute the gradients with respect to each weight
                batch_loss.backward()

                # Update the weights using the compute gradients
                optimizer.step()

                # Record the loss from this sample to keep track of progress.
                train_losses.append(batch_loss.item())

            print("Epoch: {}/{}".format((e+1), num_epochs),
                "Training Loss: {:.4f}".format(np.mean(train_losses)))

            self.eval()  # Switch model to evaluation mode

            dev_losses_epoch = []
            
            for dev_batch in dev_dataloader:
                dev_output = self(dev_batch['input_ids'])
                dev_loss = loss_fn(dev_output, dev_batch['label'])

                # Save the loss on the dev set
                dev_losses_epoch.append(dev_loss.item())
                        
            dev_losses.append(np.mean(dev_losses_epoch))
                    
            print("Epoch: {}/{}".format((e+1), num_epochs),
                "Validation Loss: {:.4f}".format(dev_losses[-1]) )

        return dev_losses

def predict_nn(trained_model, data_loader):

    trained_model.eval()

    pred_labs = []  # predicted labels to return
    
    for batch in data_loader:
        test_output = trained_model(batch['input_ids'])
        predicted_labels = test_output.argmax(1)
        pred_labs.extend(predicted_labels.tolist())
    
    return pred_labs

vocab_size = len(tokenizer) # max(tokenizer.get_vocab().values()) + 1
nn_classifier_model = FFTextClassifier(vocab_size, sequence_length, num_classes)
dev_losses = nn_classifier_model.run_training(train_loader, val_loader)

predictions_nn = predict_nn(nn_classifier_model, val_loader)

pd.DataFrame(predictions_nn).to_csv("./outputs/12_nn_preds.csv")
accuracy_nn = accuracy_score(val_labels, predictions_nn)
print(accuracy_nn)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch: 1/10 Training Loss: 1.0697
Epoch: 1/10 Validation Loss: 1.1019
Epoch: 2/10 Training Loss: 0.4479
Epoch: 2/10 Validation Loss: 1.3852
Epoch: 3/10 Training Loss: 0.1401
Epoch: 3/10 Validation Loss: 1.5641
Epoch: 4/10 Training Loss: 0.0668
Epoch: 4/10 Validation Loss: 2.1656
Epoch: 5/10 Training Loss: 0.0408
Epoch: 5/10 Validation Loss: 2.1286
Epoch: 6/10 Training Loss: 0.0732
Epoch: 6/10 Validation Loss: 2.1811
Epoch: 7/10 Training Loss: 0.0662
Epoch: 7/10 Validation Loss: 2.2500
Epoch: 8/10 Training Loss: 0.0845
Epoch: 8/10 Validation Loss: 2.1193
Epoch: 9/10 Training Loss: 0.0547
Epoch: 9/10 Validation Loss: 2.2320
Epoch: 10/10 Training Loss: 0.0256
Epoch: 10/10 Validation Loss: 2.3959
0.49


In [26]:
chosen_document = train_texts[np.random.randint(len(train_texts))]

## Transformer Embeddings and Classification

Transformers provide powerful contextual representations of text. In this section, we explore how pretrained models can be used for **document embeddings, similarity search, and classification**.

### Document Embeddings with BERT
We use the pretrained model "prajjwal1/bert-tiny" to obtain a dense vector representation of a document. The process is:
1. **Tokenization:** Convert text into token IDs with padding/truncation.
2. **Model Forward Pass:** Feed tokens into BERT to obtain hidden states.
3. **CLS Token Representation:** Use the embedding of the [CLS] token as the document representation.

In [28]:
set_seed(seed_number)

from transformers import AutoTokenizer, AutoModel
import torch

doc_emb = []  # use this variable to store the document embedding

# Loading the tiny BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
model = AutoModel.from_pretrained("prajjwal1/bert-tiny")

# choosing a document to embed
doc_chosen = val_texts[25]

# Tokenize with appropriate padding/truncation
inputs = tokenizer(
    doc_chosen,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)

# Get model output (last hidden states)
with torch.no_grad():
    outputs = model(**inputs)

# Use the CLS token representation as the embedding
cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: (1, hidden_size)

# converting to list format
doc_emb = cls_embedding.squeeze().tolist()

pd.DataFrame(doc_emb).to_csv('./outputs/13a_sen_emb.csv')
print(doc_emb)

[-0.6635202169418335, 1.1119393110275269, -3.731729507446289, -0.405616819858551, 0.21967701613903046, 0.16706888377666473, -0.4551953375339508, 0.5666081309318542, -1.4295541048049927, 0.5867826342582703, 1.0406386852264404, 1.1555777788162231, -0.45889753103256226, 0.2726829946041107, 0.5163236856460571, -1.219048261642456, -0.05327729880809784, -0.9215549230575562, -0.9084565043449402, 1.1080267429351807, -0.9565761685371399, 0.26851335167884827, 1.6170274019241333, 0.8518118262290955, 2.523898124694824, -1.3279645442962646, -0.9461399912834167, -0.9447389841079712, -0.7129529118537903, 1.0167145729064941, 0.04164308309555054, -2.053234815597534, -0.6066209077835083, -0.49764496088027954, -1.2931647300720215, 0.15455617010593414, -0.37701839208602905, 0.9184364676475525, -2.194016933441162, 0.10043208301067352, 0.12830913066864014, -0.2199392020702362, 0.13992546498775482, -3.127922296524048, -0.3089992105960846, -1.6172096729278564, -1.6550774574279785, 0.7304016947746277, -0.53533

This embedding captures the semantic meaning of the document in a compact vector.

### Document Similarity Search
To find the most similar document in the validation set, we compute embeddings for all documents and compare them using **cosine similarity**. Cosine similarity considers the **direction** of vectors, not their magnitude, making it ideal for text embeddings.

In [30]:
set_seed(seed_number)

from sklearn.metrics.pairwise import cosine_similarity

most_similar_doc = ""  # use this variable to store the most similar document

# Get embedding for the chosen document
chosen_index = 25
chosen_document = val_texts[chosen_index]
inputs = tokenizer(chosen_document, return_tensors="pt", padding="max_length", truncation=True, max_length=128)

with torch.no_grad():
    chosen_output = model(**inputs)
chosen_emb = chosen_output.last_hidden_state[:, 0, :].squeeze().numpy().reshape(1, -1)

# Compute embeddings for all other val_texts
max_sim = -1
for idx, doc in enumerate(val_texts):
    if idx == chosen_index:
        continue  # skip the chosen doc itself

    inputs = tokenizer(doc, return_tensors="pt", padding="max_length", truncation=True, max_length=128)

    with torch.no_grad():
        output = model(**inputs)
    emb = output.last_hidden_state[:, 0, :].squeeze().numpy().reshape(1, -1)

    sim = cosine_similarity(chosen_emb, emb)[0][0]
    if sim > max_sim:
        max_sim = sim
        most_similar_doc = doc


pd.DataFrame([chosen_document, most_similar_doc]).to_csv("./outputs/13b_most_similar.csv")
print(chosen_document)
print(most_similar_doc)

Additional climate variables and related environmental stressors are known to affect production but were assessed more broadly due to data and evidence limitations. These parameters include fire, cyclones, sea level rise, pests and diseases. As a result, our modelling of physical climate risk may understate the potential impact of climate change.
■ In addition, failure to adequately prepare for the potential impacts of climate change may have a negative impact on our financial position or our ability to operate. Potential impacts may be direct or indirect and may include business losses or disruption resulting from extreme weather conditions; the impact of changes in legal or regulatory framework made to address climate change; or increased mortality or morbidity resulting from environmental damage or climate change.


This allows us to identify documents with **similar climate sentiment framing**, useful for clustering or retrieval tasks.

### Transformer‑Based Classification

Finally, we fine‑tune "prajjwal1/bert-tiny" as a classifier. Using HuggingFace's `AutoModelForSequenceClassification` and `Trainer API`:
* **Tokenizer** prepares input sequences.
* **Trainer** handles training, evaluation, and logging.
* **Cross‑entropy** loss is used for multi‑class classification.

In [32]:

set_seed(seed_number)

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np

predictions_bert = []  # use this variable to store the predicted labels for the validation set

# Load tokenizer and classification model
tokenizer = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
model = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=num_classes)

# Tokenization function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Prepare train and val datasets for HuggingFace Trainer
train_data = {"text": train_texts.tolist(), "label": train_labels.tolist()}
val_data = {"text": val_texts.tolist(), "label": val_labels.tolist()}

train_dataset = Dataset.from_dict(train_data)
val_dataset = Dataset.from_dict(val_data)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Training arguments
training_args = TrainingArguments(
    output_dir="./outputs/bert_tiny_classifier",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=False,
    logging_dir="./logs"
)

# Define metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = (predictions == labels).mean()
    return {"accuracy": acc}

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Predict on validation set
predictions_raw = trainer.predict(val_dataset)
predictions_bert = predictions_raw.predictions.argmax(axis=1).tolist()

pd.DataFrame(predictions_bert).to_csv('./outputs/13c_bert_preds.csv')
accuracy_tinybert = accuracy_score(val_dataset["label"], predictions_bert)
print(accuracy_tinybert)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,1.081000,1.050497,0.460000
2,1.031500,1.009034,0.550000
3,0.988100,0.962680,0.610000
4,0.949300,0.933541,0.685000
5,0.905500,0.888262,0.705000
6,0.867300,0.855473,0.740000
7,0.834000,0.832050,0.735000
8,0.815000,0.817317,0.750000
9,0.800600,0.803909,0.750000
10,0.791400,0.801755,0.750000


0.75
